# 👺 Mistral 7B v0.3 – Compliance Fine-Tuning (Google Colab)

**Before running:** Make sure your runtime is set to **T4 GPU**.
Go to **Runtime → Change runtime type → T4 GPU** then click Save.

---
## Step 1: Install Dependencies

In [ ]:
%%capture
# Install Unsloth (Colab-specific build) and its dependencies
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl datasets transformers accelerate bitsandbytes

## Step 2: Upload & Extract Your Dataset

You need to zip your `dataset_2.1_rl` folder and upload it. Run the cell below to do it interactively.

In [ ]:
from google.colab import files
import os

print("Please upload your dataset_2.1_rl.zip file now:")
uploaded = files.upload()

# Get the uploaded filename
if not uploaded:
    print("❌ No file uploaded!")
else:
    zip_filename = list(uploaded.keys())[0]
    print(f"Uploaded: {zip_filename}")

    # Unzip the dataset
    !unzip -q "{zip_filename}" -d .

    # Verify it extracted correctly
    dataset_folder = "dataset_2.1_rl"
    if os.path.isdir(dataset_folder):
        file_count = len([f for f in os.listdir(dataset_folder) if f.endswith('.json')])
        print(f"✅ Dataset extracted! Found {file_count} JSON files in '{dataset_folder}'")
    else:
        print(f"⚠️ Could not find folder '{dataset_folder}'. Checking top-level...")
        file_count = len([f for f in os.listdir('.') if f.endswith('.json')])
        if file_count > 0:
            print(f"✅ Found {file_count} JSON files in current directory. Setting DATA_DIR to '.'")
            DATA_DIR = "."
        else:
            print("❌ No JSON files found. Please check the zip structure.")

## Step 3: Configuration & Hyperparameters

In [ ]:
import os

# --- Dataset folder ---
if 'DATA_DIR' not in locals():
    DATA_DIR = "dataset_2.1_rl"

# --- Model ---
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3" 

OUTPUT_DIR = "./compliance_mistral_model"

# --- Training Hyperparameters ---
MAX_SEQ_LENGTH = 4096   # T4 has 16GB so 4096 should be fine; lower to 2048 if OOM
LOAD_IN_4BIT   = True   # 4-bit QLoRA: essential for fitting large models in 16GB
BATCH_SIZE     = 2      
GRADIENT_ACCUMULATION = 4
EPOCHS         = 3
LEARNING_RATE  = 2e-4

print("✅ Configuration set.")
print(f"   Model      : {MODEL_NAME}")
print(f"   Data dir   : {DATA_DIR}")
print(f"   Seq length : {MAX_SEQ_LENGTH}")
print(f"   Batch size : {BATCH_SIZE}")

## Step 4: Load & Format the Dataset

In [ ]:
import json
from datasets import Dataset

def load_and_format_dataset(data_dir):
    """
    Reads all JSON files from `data_dir`.
    Formats them for Mistral-7B-Instruct-v0.3 chat-style fine-tuning.
    """
    conversations = []

    for filename in os.listdir(data_dir):
        if not filename.endswith(".json"):
            continue

        filepath = os.path.join(data_dir, filename)
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)

            instruction = data.get("instruction", "")
            user_input  = data.get("input", "")
            final_text  = data.get("output", {}).get("final_text", "")

            if not instruction or not final_text:
                continue

            convo = [
                {"role": "system",    "content": "You are a compliance-focused insurance documentation assistant."},
                {"role": "user",      "content": f"{instruction}\n\n{user_input}"},
                {"role": "assistant", "content": final_text}
            ]
            conversations.append({"messages": convo})

        except Exception as e:
            print(f"⚠️ Error loading {filepath}: {e}")

    hf_dataset = Dataset.from_list(conversations)
    print(f"✅ Loaded {len(hf_dataset)} training examples.")
    return hf_dataset

dataset = load_and_format_dataset(DATA_DIR)
print(dataset)

## Step 5: Load Model & Tokenizer with Unsloth

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

print("Loading Mistral model and tokenizer...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype         = None,       
    load_in_4bit  = LOAD_IN_4BIT,
)

# Apply Mistral chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "mistral",
)

print(f"✅ Mistral model loaded: {MODEL_NAME}")

## Step 6: Apply Chat Template to Dataset

In [ ]:
def apply_chat_template(examples):
    """Formats messages into the Mistral chat prompt format."""
    texts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in examples["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(apply_chat_template, batched=True)
print("✅ Chat template applied to dataset.")

## Step 7: Apply LoRA Adapters

In [ ]:
print("Applying LoRA adapters...")

model = FastLanguageModel.get_peft_model(
    model,
    r                  = 16,
    target_modules     = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
    lora_alpha         = 16,
    lora_dropout       = 0,
    bias               = "none",
    use_gradient_checkpointing = "unsloth",
    random_state       = 3407,
)

print("✅ LoRA adapters applied.")

## Step 8: Train the Model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print("Initializing SFTTrainer...")

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = dataset,
    dataset_text_field = "text",
    max_seq_length  = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing         = True,   
    args = TrainingArguments(
        per_device_train_batch_size   = BATCH_SIZE,
        gradient_accumulation_steps   = GRADIENT_ACCUMULATION,
        warmup_steps                  = 100,
        num_train_epochs              = EPOCHS,
        learning_rate                 = LEARNING_RATE,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps                 = 10,
        optim                         = "adamw_8bit",
        weight_decay                  = 0.01,
        lr_scheduler_type             = "linear",
        seed                          = 3407,
        output_dir                    = "lora_checkpoints",
        report_to                     = "none",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print("\n✅ Training complete!")

## Step 9: Save the Fine-Tuned Model

In [ ]:
print(f"Saving LoRA adapters to '{OUTPUT_DIR}'...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

merged_dir = f"{OUTPUT_DIR}_merged"
print(f"Saving merged 16-bit model to '{merged_dir}'...")
model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")

print(f"✅ Models saved successfully.")

## Step 10: Run Inference

In [ ]:
FastLanguageModel.for_inference(model)

insurance_article = """
If you crash your car, we will pay you money. You need to call us within 2 days or we won't pay.
Also, don't lie to us because if we find out, we keep the money.
"""

messages = [
    {"role": "system",  "content": "You are a compliance-focused insurance documentation assistant."},
    {"role": "user",    "content": f"Rewrite the content to meet insurance regulatory and compliance standards.\n\n{insurance_article}"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize              = True,
    add_generation_prompt = True,
    return_tensors        = "pt",
).to("cuda")

print("Generating compliant rewrite with Mistral...")
outputs = model.generate(
    input_ids    = inputs,
    max_new_tokens = 1024,
    use_cache    = True,
    temperature  = 0.3,
    top_p        = 0.9,
)

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("\n--- MISTRAL REWRITTEN COMPLIANT ARTICLE ---")
print(response)